# Business24 — Khôi phục backup và mở API
**Không training.** Chạy lần lượt cell 1 → 7 trong runtime **L4 24 GB hoặc A100 40 GB / High RAM**.
L4 dùng nguyên BF16, batch 1, không quantize; tốc độ/chi phí hoàn thành cần đo thực tế.
Merge trên CPU vẫn cần ít nhất 32 GiB RAM hệ thống khả dụng và 60 GiB ổ trống.
Khi có Base URL, gửi agent để đo Train/Public; giữ runtime hoạt động trong lượt đo.
120 units là ngân sách người dùng báo; notebook không ước lượng số giờ hoặc tự train.

**Chỉ cần 1 file upload:** `business24-adapter-backup.zip` (~154 MB).
File trên máy: `/home/pt/AI Challenge/business24-adapter-backup.zip`.
Windows/WSL có thể mở `\\wsl.localhost\Ubuntu\home\pt\AI Challenge` nếu distro tên Ubuntu;
tên distro khác thì dùng đường dẫn tương ứng, hoặc tải ZIP về Downloads để chọn dễ hơn.
Không cần upload training bundle, `training_manifest.json` riêng hay file `.py`.

Nếu mất runtime: mở lại notebook này, chạy 1 → 7, upload lại ZIP; model gốc sẽ tải và merge lại.
Adapter đã train xong nên **không train lại**. Nếu runtime còn model hoàn chỉnh, cell 5 sẽ dùng lại.
Notebook không tạo/nộp submission và không thao tác portal.

In [ ]:
#@title 1. Kiểm tra GPU/RAM và chuẩn bị công cụ (mở code nếu cần xem chi tiết)
EMBEDDED_SOURCES = {'business24_recovery.py': '"""Verified backup recovery and managed serving; no training dependencies."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport tempfile\nimport time\nimport zipfile\n\nBASE_MODEL = \'Qwen/Qwen2.5-Coder-7B-Instruct\'\nREVISION = \'c03e6d358207e414f1eca0bb1891e29f1db0e242\'\nDATA_SHA = \'55c2918798e68452be79cf69bc9244311d900af60533554d2fa436bb414a908e\'\nSPLIT_SHA = \'ba0a70ea94ecf36c6d74b308ba7bcf29358f1ec693fc6cf73167c0e59b6c4ffa\'\nBACKUP_SHA = \'65914f4369f8409854b75c220ef3999077d4f73b2761c4c48dc71dbde7604965\'\n\n\ndef check_inference_gpu(csv, *, managed_api_running=False):\n    """Validate CUDA device0, matching the API\'s explicit device_map={\'\':0}."""\n    if not csv.strip():\n        raise ValueError(\'Chưa có GPU. Chọn L4 24GB hoặc A100 / High RAM.\')\n    name, total, free = (part.strip() for part in csv.strip().splitlines()[0].rsplit(\',\', 2))\n    total, free = float(total), float(free)\n    if total < 22000:\n        raise ValueError(\'Cần GPU tối thiểu L4 24GB; GPU device0 hiện không đủ VRAM cho BF16.\')\n    if total < 35000 and free < 20000 and not managed_api_running:\n        raise ValueError(\'GPU 24GB cần ít nhất 20000 MiB VRAM trống trước nạp model; dừng process GPU khác.\')\n    return {\'name\':name, \'total_mib\':total, \'free_mib\':free, \'device\':0}\n\n\ndef ensure_pip(python, bootstrap_python, run_logged):\n    """A Python executable may exist in a venv whose pip seed failed."""\n    command = [str(python), \'-m\', \'pip\', \'--version\']\n    if subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).returncode == 0:\n        return\n    # pip >=22.3 can manage a target environment even when it has no pip.\n    run_logged([str(bootstrap_python), \'-m\', \'pip\', \'--python\', str(python), \'install\', \'pip\'], \'install.log\')\n    if subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).returncode:\n        raise RuntimeError(\'pip trong venv chưa hoạt động; dừng và gửi install.log, không chạy cell tiếp theo\')\n\n\ndef check_manifest(directory, verify_data=True):\n    directory = Path(directory)\n    manifest = json.loads((directory / \'training_manifest.json\').read_text())\n    expected = {\'training_complete\': True, \'examples\': 633, \'overlap_businesses\': [],\n                \'base_model\': BASE_MODEL, \'base_revision\': REVISION,\n                \'data_sha256\': DATA_SHA, \'split_sha256\': SPLIT_SHA}\n    for key, value in expected.items():\n        if manifest.get(key) != value:\n            raise ValueError(f\'Manifest mismatch: {key}\')\n    if verify_data:\n        for name, key in [(\'training_data.jsonl\', \'data_sha256\'), (\'split.json\', \'split_sha256\')]:\n            if hashlib.sha256((directory / name).read_bytes()).hexdigest() != manifest[key]:\n                raise ValueError(f\'Corrupted {name}\')\n        for name in [\'adapter_model.safetensors\', \'adapter_config.json\', \'tokenizer.json\', \'tokenizer_config.json\']:\n            if not (directory / name).is_file() or not (directory / name).stat().st_size:\n                raise ValueError(f\'Missing adapter file: {name}\')\n    return manifest\n\n\ndef restore_backup(archive, target, expected_sha=BACKUP_SHA):\n    target = Path(target)\n    if target.exists():\n        return check_manifest(target)\n    archive = Path(archive)\n    with archive.open(\'rb\') as stream:\n        if hashlib.file_digest(stream, \'sha256\').hexdigest() != expected_sha:\n            raise ValueError(\'Backup SHA256 mismatch; upload đúng business24-adapter-backup.zip\')\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        if z.testzip() is not None:\n            raise ValueError(\'Backup ZIP CRC failed\')\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise ValueError(\'Duplicate ZIP entries\')\n        for name in z.namelist():\n            if Path(name).is_absolute() or \'..\' in Path(name).parts:\n                raise ValueError(\'Invalid ZIP path\')\n        with tempfile.TemporaryDirectory(prefix=\'business24-restore-\', dir=target.parent) as temporary:\n            staged = Path(temporary) / \'adapter\'\n            z.extractall(staged)\n            manifest = check_manifest(staged)\n            staged.rename(target)\n    return manifest\n\n\ndef check_merged(directory):\n    directory = Path(directory)\n    check_manifest(directory, verify_data=False)\n    merge = json.loads((directory / \'merge_manifest.json\').read_text())\n    for key, expected in {\'merge_complete\': True, \'base_model\': BASE_MODEL, \'base_revision\': REVISION,\n                          \'data_sha256\': DATA_SHA, \'split_sha256\': SPLIT_SHA}.items():\n        if merge.get(key) != expected:\n            raise ValueError(f\'Merge manifest mismatch: {key}\')\n    weights = merge.get(\'weights_files\', [])\n    if not weights or any(Path(name).name != name for name in weights):\n        raise ValueError(\'Invalid weights list\')\n    if any(not (directory / name).is_file() for name in weights):\n        raise ValueError(\'Missing merged weights shard\')\n    if any((directory / name).stat().st_size == 0 for name in weights):\n        raise ValueError(\'Empty merged weights shard\')\n    if sum((directory / name).stat().st_size for name in weights) != merge.get(\'weights_bytes\'):\n        raise ValueError(\'Incomplete merged weights size\')\n    if not (directory / \'config.json\').is_file() or not (directory / \'tokenizer_config.json\').is_file():\n        raise ValueError(\'Missing merged config/tokenizer\')\n    index = directory / \'model.safetensors.index.json\'\n    if index.is_file() and set(json.loads(index.read_text())[\'weight_map\'].values()) != set(weights):\n        raise ValueError(\'Merged weights index mismatch\')\n    return merge\n\n\ndef process_identity(pid):\n    """Linux PID start time distinguishes an owned process from a reused PID."""\n    try:\n        fields = Path(f\'/proc/{pid}/stat\').read_text().rsplit(\') \', 1)[1].split()\n        return None if fields[0] == \'Z\' else fields[19]\n    except (OSError, IndexError):\n        return None\n\n\ndef managed_process(work):\n    state = Path(work) / \'api-process.json\'\n    if state.is_file():\n        info = json.loads(state.read_text())\n        if process_identity(info[\'pid\']) == info[\'start_time\']:\n            return info\n    return None\n\n\ndef stop_api(work):\n    info = managed_process(work)\n    if info:\n        # Only the session started here (start_new_session=True) may be stopped.\n        if os.getpgid(info[\'pid\']) != info[\'pid\']:\n            raise RuntimeError(\'Process group mismatch; không dừng process lạ\')\n        os.killpg(info[\'pid\'], signal.SIGTERM)\n        for _ in range(50):\n            if process_identity(info[\'pid\']) is None:\n                break\n            time.sleep(0.1)\n        if process_identity(info[\'pid\']) is not None:\n            raise RuntimeError(\'API chưa dừng; kiểm tra trước khi mở lại\')\n    (Path(work) / \'api-process.json\').unlink(missing_ok=True)\n\n\ndef start_api(python, api_script, work, restart=False):\n    work = Path(work)\n    if restart:\n        stop_api(work)\n    info = managed_process(work)\n    if info:\n        return info[\'pid\']\n    work.mkdir(parents=True, exist_ok=True)\n    runner = work / \'serve_business24.py\'\n    runner.write_text(\n        \'import runpy, threading, signal\\n\'\n        f\'namespace = runpy.run_path({str(Path(api_script).resolve())!r})\\n\'\n        \'reopen = threading.Event()\\n\'\n        \'signal.signal(signal.SIGUSR1, lambda *_: reopen.set())\\n\'\n        \'namespace["start_business24_api"]()\\n\'\n        \'while True:\\n\'\n        \'    reopen.wait()\\n\'\n        \'    reopen.clear()\\n\'\n        \'    state = namespace["start_business24_api"].__globals__\\n\'\n        \'    tunnel = state.get("_business24_tunnel")\\n\'\n        \'    if tunnel is not None and tunnel.poll() is None:\\n\'\n        \'        tunnel.terminate()\\n\'\n        \'        tunnel.wait(timeout=10)\\n\'\n        \'    state["_business24_tunnel"] = None\\n\'\n        \'    namespace["start_business24_api"]()\\n\'\n    )\n    with (work / \'api.log\').open(\'w\') as log:\n        proc = subprocess.Popen([str(python), \'-u\', str(runner)], stdout=log,\n                                stderr=subprocess.STDOUT, start_new_session=True)\n    identity = process_identity(proc.pid)\n    if identity is None:\n        raise RuntimeError(\'API đã thoát; xem api.log\')\n    state = {\'pid\': proc.pid, \'start_time\': identity}\n    (work / \'api-process.json\').write_text(json.dumps(state))\n    return proc.pid\n\n\ndef reopen_tunnel(work):\n    info = managed_process(work)\n    if info is None:\n        raise RuntimeError(\'API chưa chạy; chạy cell mở API trước\')\n    os.kill(info[\'pid\'], signal.SIGUSR1)\n', 'business24_merge_cell.py': "# Dán toàn bộ nội dung này vào cell MỚI cuối notebook 03, sau khi đã backup adapter.\n# Chạy trong runtime 03 hiện tại; không restart hoặc chạy lại cell training.\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\n\nADAPTER = Path('/content/qwen2.5-coder-7b-lora-business24')\nMERGED = Path('/content/qwen2.5-coder-7b-datamate-business24')\nmanifest = json.loads((ADAPTER / 'training_manifest.json').read_text())\nassert manifest['training_complete'] is True\nassert manifest['examples'] == 633 and not manifest['overlap_businesses']\nassert manifest['base_model'] == 'Qwen/Qwen2.5-Coder-7B-Instruct'\nassert manifest['base_revision'] == 'c03e6d358207e414f1eca0bb1891e29f1db0e242'\nassert manifest['data_sha256'] == '55c2918798e68452be79cf69bc9244311d900af60533554d2fa436bb414a908e'\nfor filename, key in [('training_data.jsonl', 'data_sha256'), ('split.json', 'split_sha256')]:\n    assert hashlib.sha256((ADAPTER / filename).read_bytes()).hexdigest() == manifest[key]\nif MERGED.exists():\n    raise FileExistsError(f'Không ghi đè thư mục đã có: {MERGED}')\n\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nfrom peft import PeftModel\n\nprint('Đang nạp đúng Qwen revision trên RAM hệ thống...', flush=True)\nbase = AutoModelForCausalLM.from_pretrained(\n    manifest['base_model'], revision=manifest['base_revision'],\n    torch_dtype=torch.bfloat16, device_map='cpu', trust_remote_code=False,\n)\nmodel_with_adapter = PeftModel.from_pretrained(base, str(ADAPTER), is_trainable=False)\nprint('Đang gộp adapter...', flush=True)\nmerged_model = model_with_adapter.merge_and_unload(safe_merge=True)\nmerged_model.eval()\nmerged_model.config.use_cache = True\nmerged_model.save_pretrained(str(MERGED), safe_serialization=True, max_shard_size='5GB')\ntokenizer = AutoTokenizer.from_pretrained(str(ADAPTER), trust_remote_code=False)\ntokenizer.save_pretrained(str(MERGED))\n(MERGED / 'training_manifest.json').write_bytes((ADAPTER / 'training_manifest.json').read_bytes())\nweights = list(MERGED.glob('*.safetensors'))\nassert weights and (MERGED / 'config.json').is_file()\nmerge_report = {\n    'merge_complete': True,\n    'base_model': manifest['base_model'], 'base_revision': manifest['base_revision'],\n    'data_sha256': manifest['data_sha256'], 'split_sha256': manifest['split_sha256'],\n    'weights_bytes': sum(p.stat().st_size for p in weights),\n    'weights_files': sorted(p.name for p in weights),\n    'validation_pa': None,\n}\n(MERGED / 'merge_manifest.json').write_text(json.dumps(merge_report, indent=2))\ndel merged_model, model_with_adapter, base\ngc.collect()\nif torch.cuda.is_available():\n    torch.cuda.empty_cache()\nprint('MERGE HOÀN TẤT:', MERGED, flush=True)\nprint('Bước tiếp theo: phục vụ model từ đường dẫn mới này và đánh giá 6 business giữ riêng.')\n", 'business24_api_cell.py': "# Copy toàn bộ file vào cell MỚI trong notebook 03, sau cell merge.\n# Không restart runtime và không chạy lại training.\nimport hashlib\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport threading\nimport urllib.request\n\nMODEL_ID = 'datamate-business24'\nMODEL_DIR = Path('/content/qwen2.5-coder-7b-datamate-business24')\nEXPECTED_DATA = '55c2918798e68452be79cf69bc9244311d900af60533554d2fa436bb414a908e'\nEXPECTED_REVISION = 'c03e6d358207e414f1eca0bb1891e29f1db0e242'\n\n\ndef make_server(generate, provenance, port=8000):\n    lock = threading.Lock()\n\n    class Handler(BaseHTTPRequestHandler):\n        def log_message(self, *_):\n            pass  # Do not print prompts or generated content into HTTP logs.\n\n        def reply(self, status, payload):\n            body = json.dumps(payload, ensure_ascii=False, allow_nan=False).encode()\n            self.send_response(status)\n            self.send_header('Content-Type', 'application/json; charset=utf-8')\n            self.send_header('Content-Length', str(len(body)))\n            self.end_headers()\n            self.wfile.write(body)\n\n        def do_GET(self):\n            if self.path == '/v1/models':\n                self.reply(200, {'data':[{'id':MODEL_ID}]})\n            elif self.path == '/v1/model-provenance':\n                self.reply(200, provenance)\n            else:\n                self.reply(404, {'error':'Unknown path'})\n\n        def do_POST(self):\n            if self.path != '/v1/chat/completions':\n                self.reply(404, {'error':'Unknown path'})\n                return\n            try:\n                self.connection.settimeout(30)\n                size = int(self.headers.get('Content-Length','0'))\n                if not 0 < size <= 131072:\n                    raise ValueError('Invalid request size')\n                payload = json.loads(self.rfile.read(size))\n                if not isinstance(payload,dict) or payload.get('model') != MODEL_ID:\n                    raise ValueError('Use model datamate-business24')\n                if payload.get('temperature',0) != 0 or payload.get('seed',42) != 42 or payload.get('stream',False):\n                    raise ValueError('Only temperature=0, seed=42, non-streaming requests are supported')\n                messages = payload.get('messages')\n                limit = payload.get('max_tokens',1024)\n                if type(limit) is not int or not 1 <= limit <= 2048:\n                    raise ValueError('max_tokens must be 1..2048')\n                if not isinstance(messages,list) or not 1 <= len(messages) <= 16:\n                    raise ValueError('Expected 1..16 messages')\n                for m in messages:\n                    if (not isinstance(m,dict) or m.get('role') not in ('system','user','assistant')\n                            or not isinstance(m.get('content'),str) or len(m['content']) > 48000):\n                        raise ValueError('Invalid message')\n            except (ValueError, TypeError, OSError) as exc:\n                self.reply(400, {'error':str(exc)})\n                return\n            if not lock.acquire(blocking=False):\n                self.reply(429, {'error':'Model busy; retry after current request'})\n                return\n            try:\n                result = generate(messages,limit)\n                self.reply(200, {'model':MODEL_ID,'choices':[{'index':0,'message':{'role':'assistant','content':result['text']},\n                                'finish_reason':result['finish_reason']}],\n                                'usage':{'prompt_tokens':result['prompt_tokens'],'completion_tokens':result['completion_tokens']}})\n            except ValueError as exc:\n                self.reply(400, {'error':str(exc)})\n            except Exception as exc:\n                print('Inference error:',type(exc).__name__,str(exc),flush=True)\n                self.reply(500, {'error':'Model generation failed; inspect Colab output'})\n            finally:\n                lock.release()\n\n    server = ThreadingHTTPServer(('127.0.0.1',port),Handler)\n    server.daemon_threads = True\n    return server\n\n\ndef tunnel_binary():\n    binary = shutil.which('cloudflared')\n    if binary:\n        return binary\n    # Pinned official release + published SHA256; no dependency on GitHub API rate limits.\n    # https://github.com/cloudflare/cloudflared/releases/tag/2026.10.7\n    version = '2026.10.7'\n    expected_sha256 = '1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872'\n    url = f'https://github.com/cloudflare/cloudflared/releases/download/{version}/cloudflared-linux-amd64'\n    with urllib.request.urlopen(url,timeout=60) as response:\n        data = response.read(100_000_001)\n    if len(data)>100_000_000 or hashlib.sha256(data).hexdigest()!=expected_sha256:\n        raise RuntimeError('cloudflared SHA256 mismatch')\n    target = Path('/tmp/cloudflared-business24')\n    target.write_bytes(data)\n    target.chmod(0o700)\n    print('Verified cloudflared release:',version,flush=True)\n    return str(target)\n\n\ndef start_business24_api():\n    global _business24_server, _business24_tunnel\n    if globals().get('_business24_server') is None:\n        manifest = json.loads((MODEL_DIR/'training_manifest.json').read_text())\n        merge = json.loads((MODEL_DIR/'merge_manifest.json').read_text())\n        assert manifest['training_complete'] is True and merge['merge_complete'] is True\n        assert manifest['examples']==633 and not manifest['overlap_businesses']\n        assert manifest['data_sha256']==merge['data_sha256']==EXPECTED_DATA\n        assert manifest['base_revision']==merge['base_revision']==EXPECTED_REVISION\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        if not torch.cuda.is_available():\n            raise RuntimeError('Chọn runtime L4 24GB hoặc A100 GPU trước khi chạy cell này')\n        if not torch.cuda.is_bf16_supported():\n            raise RuntimeError('GPU không hỗ trợ BF16; dừng trước khi nạp model')\n        free_bytes, total_bytes = torch.cuda.mem_get_info(0)\n        if total_bytes < 22000 * 1024**2 or (total_bytes < 35000 * 1024**2 and free_bytes < 20000 * 1024**2):\n            raise RuntimeError('VRAM device0 không đủ cho BF16; cần GPU 24GB còn trống trước nạp model')\n        print('Đang nạp MODEL BUSINESS24 MỚI vào GPU...',flush=True)\n        tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR),trust_remote_code=False)\n        model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR),torch_dtype=torch.bfloat16,\n                                                    device_map={'':0},trust_remote_code=False,\n                                                    low_cpu_mem_usage=True,attn_implementation='sdpa')\n        model.eval()\n        model.config.use_cache = True\n        torch.manual_seed(42)\n\n        def generate(messages,max_tokens):\n            text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)\n            inputs = tokenizer(text,return_tensors='pt').to('cuda:0')\n            prompt_tokens = inputs.input_ids.shape[1]\n            if prompt_tokens+max_tokens>8192:\n                raise ValueError('Prompt + requested completion exceeds 8192 tokens')\n            with torch.inference_mode():\n                output = model.generate(**inputs,max_new_tokens=max_tokens,do_sample=False,\n                                        pad_token_id=tokenizer.eos_token_id)\n            new_tokens = output[0,prompt_tokens:]\n            return {'text':tokenizer.decode(new_tokens,skip_special_tokens=True),\n                    'prompt_tokens':prompt_tokens,'completion_tokens':len(new_tokens),\n                    'finish_reason':'length' if len(new_tokens)>=max_tokens else 'stop'}\n\n        torch.cuda.reset_peak_memory_stats(0)\n        generate([{'role':'user','content':'Trả lời OK.'}],1)\n        torch.cuda.synchronize(0)\n        runtime = {'gpu':torch.cuda.get_device_name(0),'dtype':'bfloat16',\n                   'attention':'sdpa','batch_size':1,'max_context_tokens':8192,\n                   'total_vram_gib':round(total_bytes/1024**3,2),\n                   'startup_peak_allocated_gib':round(torch.cuda.max_memory_allocated(0)/1024**3,2)}\n        print('MODEL LOAD + 1-TOKEN SMOKE OK:',json.dumps(runtime),flush=True)\n        _business24_server = make_server(generate,{'model':MODEL_ID,'model_dir':str(MODEL_DIR),\n                                                 **manifest,'merge_complete':True,'runtime':runtime})\n        threading.Thread(target=_business24_server.serve_forever,daemon=True).start()\n    with urllib.request.urlopen('http://127.0.0.1:8000/v1/models',timeout=5) as response:\n        assert json.load(response)['data'][0]['id']==MODEL_ID\n    if globals().get('_business24_tunnel') is not None and _business24_tunnel.poll() is None:\n        print('Tunnel đã chạy. Dùng URL được in ở lần chạy trước; không restart cell.',flush=True)\n        return\n    proc = subprocess.Popen([tunnel_binary(),'--no-autoupdate','tunnel','--url','http://127.0.0.1:8000'],\n                            stdout=subprocess.DEVNULL,stderr=subprocess.PIPE,text=True)\n    _business24_tunnel = proc\n    ready = threading.Event()\n    found = []\n\n    def watch_tunnel():\n        for line in proc.stderr:\n            m = re.search(r'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com',line)\n            if m and not found:\n                found.append(m.group(0))\n                ready.set()\n            if 'ERR' in line:\n                print(line.strip(),flush=True)\n        ready.set()\n\n    threading.Thread(target=watch_tunnel,daemon=True).start()\n    if not ready.wait(60) or not found:\n        raise RuntimeError('Chưa nhận được URL tunnel; giữ runtime và gửi output này cho agent')\n    print('\\nAPI BUSINESS24 SẴN SÀNG',flush=True)\n    print('Base URL:',found[0]+'/v1',flush=True)\n    print('Model:',MODEL_ID,flush=True)\n    print('Gửi Base URL này cho agent. Giữ runtime hoạt động; không cần chạy lại training.',flush=True)\n\n\nif __name__=='__main__':\n    start_business24_api()\n"}

from pathlib import Path
import importlib.util, json, os, shutil, subprocess, sys, time

WORK = Path('/content/business24-recovery')
ADAPTER = Path('/content/qwen2.5-coder-7b-lora-business24')
MERGED = Path('/content/qwen2.5-coder-7b-datamate-business24')
VENV = Path('/content/business24-venv')
PYTHON = VENV / 'bin/python'
if sys.version_info < (3, 11) or sys.version_info >= (3, 15):
    raise RuntimeError('Notebook cần Python 3.11–3.14. Gửi phiên bản runtime cho agent.')
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → L4 hoặc A100 GPU / High RAM.')
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'], text=True)
print('GPU:', gpu.strip())
ram_kib = {line.split(':')[0]: int(line.split()[1]) for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith(('MemTotal:', 'MemAvailable:'))}
print('RAM tổng / khả dụng (GiB):', round(ram_kib['MemTotal']/1024**2, 1), '/', round(ram_kib['MemAvailable']/1024**2, 1))
if not MERGED.exists() and ram_kib['MemAvailable'] < 32 * 1024**2:
    raise RuntimeError('RAM hệ thống khả dụng dưới 32 GiB. Chọn High RAM kể cả khi GPU là L4; chưa tải/merge model.')
free_gib = shutil.disk_usage('/content').free / 1024**3
print('Ổ đĩa trống (GiB):', round(free_gib, 1))
if not MERGED.exists() and free_gib < 60:
    raise RuntimeError('Cần ít nhất 60 GiB trống cho thư viện, base cache và merged model.')
WORK.mkdir(parents=True, exist_ok=True)
for name, source in EMBEDDED_SOURCES.items():
    (WORK / name).write_text(source)
spec = importlib.util.spec_from_file_location('business24_recovery', WORK / 'business24_recovery.py')
recovery = importlib.util.module_from_spec(spec)
spec.loader.exec_module(recovery)
GPU_PROFILE = recovery.check_inference_gpu(gpu, managed_api_running=recovery.managed_process(WORK) is not None)
print('BF16 inference profile:', GPU_PROFILE)
print('Sẵn sàng. Adapter còn:', ADAPTER.exists(), '| Merged còn:', MERGED.exists())

## Cell 2 — Môi trường riêng
Cài đúng Torch CUDA build trong training manifest và bộ merge đã dùng trước đó.
`venv` tách khỏi Gradio/Diffusers của Colab; **không pip install vào kernel hệ thống**.
Lần đầu tải thư viện có thể mất vài phút; chạy lại cell sẽ dùng môi trường đã kiểm.
Nếu venv còn Python nhưng thiếu pip, cell tự khôi phục pip vào đúng venv trước khi cài.
Không cần Restart session sau bước này. Log lưu tại `/content/business24-recovery/install.log`.

In [ ]:
#@title 2. Cài thư viện riêng và kiểm CUDA/bfloat16
def run_logged(command, filename):
    log_path = WORK / filename
    with log_path.open('a') as log:
        proc = subprocess.Popen([str(x) for x in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        status = proc.wait()
    if status:
        raise RuntimeError(f'Lệnh thất bại (exit {status}); gửi output phía trên. Log: {log_path}')

stamp = VENV / 'business24-environment.json'
expected_environment = {'torch': '2.11.0+cu130', 'transformers': '4.57.1',
                        'peft': '0.17.1', 'accelerate': '1.11.0', 'huggingface-hub': '0.36.2'}
if not stamp.is_file() or json.loads(stamp.read_text()) != expected_environment:
    if recovery.managed_process(WORK):
        raise RuntimeError('API đang chạy; không cài lại thư viện giữa lượt đo. Dừng bằng cell 8 trước.')
    if not PYTHON.exists():
        run_logged([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], 'install.log')
    recovery.ensure_pip(PYTHON, sys.executable, run_logged)
    run_logged([PYTHON, '-m', 'pip', 'install', 'torch==2.11.0+cu130', '--index-url', 'https://download.pytorch.org/whl/cu130'], 'install.log')
    run_logged([PYTHON, '-m', 'pip', 'install', 'transformers==4.57.1', 'peft==0.17.1', 'accelerate==1.11.0', 'huggingface-hub==0.36.2'], 'install.log')
    run_logged([PYTHON, '-m', 'pip', 'check'], 'install.log')
    stamp.write_text(json.dumps(expected_environment))
smoke = """import importlib.metadata as m, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
expected = EXPECTED_ENVIRONMENT
assert all(m.version(k) == v for k, v in expected.items()), 'Installed version mismatch'
assert torch.cuda.is_available(), 'CUDA không hoạt động; gửi output nvidia-smi cho agent'
assert torch.cuda.is_bf16_supported(), 'GPU không hỗ trợ bfloat16'
print('IMPORT/CUDA OK:', torch.__version__, torch.cuda.get_device_name(0))
""".replace('EXPECTED_ENVIRONMENT', repr(expected_environment))
run_logged([PYTHON, '-u', '-c', smoke], 'install.log')

## Cell 3 — Upload ZIP backup
Chọn **`business24-adapter-backup.zip`** tại `/home/pt/AI Challenge/business24-adapter-backup.zip` trên máy.
Trong Colab file được đặt ở `/content/business24-adapter-backup.zip`.
Nếu ZIP đã nằm trên Drive, có thể điền đường dẫn ở `BACKUP_ON_DRIVE`; để trống thì chọn upload từ máy.
Adapter còn và hợp lệ sẽ được dùng lại, không hỏi upload. ZIP kiểm SHA256 + CRC trước giải nén.

In [ ]:
#@title 3. Khôi phục adapter từ backup đã train xong
BACKUP_ON_DRIVE = '' #@param {type:"string"}
archive = Path('/content/business24-adapter-backup.zip')
if MERGED.exists():
    recovery.check_merged(MERGED)
    manifest = recovery.check_manifest(MERGED, verify_data=False)
    print('Merged model đã hợp lệ; không cần upload hoặc khôi phục adapter.')
elif ADAPTER.exists():
    manifest = recovery.check_manifest(ADAPTER)
    print('Adapter đã có và hợp lệ; bỏ qua upload.')
else:
    if BACKUP_ON_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        archive = Path(BACKUP_ON_DRIVE)
    elif not archive.is_file():
        from google.colab import files
        os.chdir('/content')
        uploaded = files.upload()  # CHỈ chọn business24-adapter-backup.zip
        del uploaded  # Giải phóng bản upload trong RAM.
    if not archive.is_file():
        raise FileNotFoundError('Chưa thấy business24-adapter-backup.zip đúng tên/đường dẫn; chọn lại file.')
    manifest = recovery.restore_backup(archive, ADAPTER)
print('BACKUP OK:', manifest['examples'], 'mẫu; train complete:', manifest['training_complete'])
print('Base revision:', manifest['base_revision'])

## Cell 4 — Merge một lần
Nạp base Qwen đúng revision trên RAM hệ thống rồi gộp adapter; **không training**.
Lần đầu cần tải khoảng 15 GB base weights. Log: `/content/business24-recovery/merge.log`.
Chạy lại sẽ kiểm và dùng merged model hoàn chỉnh. Thư mục dở dang sẽ bị từ chối, không tự xóa.

In [ ]:
#@title 4. Merge hoặc dùng lại model đã hoàn tất
if MERGED.exists():
    merge = recovery.check_merged(MERGED)
    print('Dùng lại merged model đã kiểm:', MERGED)
else:
    if recovery.managed_process(WORK):
        raise RuntimeError('API đang chạy; không merge trong cùng lượt đo.')
    run_logged([PYTHON, '-u', WORK / 'business24_merge_cell.py'], 'merge.log')
    merge = recovery.check_merged(MERGED)
print('MERGED OK:', round(merge['weights_bytes'] / 1024**3, 2), 'GiB weights')

## Cell 5 — Mở API và lấy URL
Mặc định dùng lại process API đang chạy, không nạp thêm model. Đợi tới khi in **Base URL** rồi gửi agent.
Nếu URL cũ mất nhưng runtime còn hoạt động: bật `REOPEN_TUNNEL`, chạy lại cell này; chỉ mở lại tunnel, giữ model trong GPU.
Nếu model/API lỗi, xem log; không lặp Run all khi chưa đọc lỗi. Không chạy notebook 02/training đồng thời trên runtime này.

In [ ]:
#@title 5. Mở API (hoặc mở lại riêng tunnel)
import re, socket, urllib.request
REOPEN_TUNNEL = False #@param {type:"boolean"}
recovery.check_merged(MERGED)
already_running = recovery.managed_process(WORK) is not None
if not already_running:
    with socket.socket() as probe:
        if probe.connect_ex(('127.0.0.1', 8000)) == 0:
            raise RuntimeError('Cổng 8000 đã có API khác. Dừng đúng API cũ trước; chưa nạp thêm model.')
log_path = WORK / 'api.log'
offset = log_path.stat().st_size if already_running and REOPEN_TUNNEL else 0
pid = recovery.start_api(PYTHON, WORK / 'business24_api_cell.py', WORK)
if already_running and REOPEN_TUNNEL:
    recovery.reopen_tunnel(WORK)
deadline = time.monotonic() + 300
printed = offset
BASE_URL = None
while time.monotonic() < deadline:
    content = log_path.read_text(errors='replace')
    if len(content) > printed:
        print(content[printed:], end='')
        printed = len(content)
    urls = re.findall(r'Base URL:\s*(https://[a-zA-Z0-9-]+\.trycloudflare\.com/v1)', content[offset:])
    if urls:
        BASE_URL = urls[-1]
        break
    if recovery.managed_process(WORK) is None:
        raise RuntimeError(f'API đã thoát. Gửi log phía trên; file {log_path}')
    time.sleep(2)
if not BASE_URL:
    raise TimeoutError(f'Chưa sẵn sàng sau 300 giây; xem {log_path}, đừng nạp thêm model.')
print('\nGỬI AGENT:', BASE_URL, '| Model: datamate-business24')
REOPEN_TUNNEL = False

## Cell 6 — Kiểm API/provenance
Cell 5 đã sinh thử 1 token trước mở API; đây chưa phải phép đo context dài hoặc chất lượng model.
Kiểm localhost trước, rồi đường dẫn công khai. Tunnel mới có thể cần ít giây để sẵn sàng.
Lỗi public sau khi local OK có thể do tunnel/network; chưa có nghĩa weights hỏng.

In [ ]:
#@title 6. Kiểm tra API và manifest
def read_provenance(base_url):
    with urllib.request.urlopen(base_url + '/model-provenance', timeout=20) as response:
        metadata = json.load(response)
    expected = {'model': 'datamate-business24', 'training_complete': True, 'merge_complete': True,
                'data_sha256': recovery.DATA_SHA, 'split_sha256': recovery.SPLIT_SHA,
                'base_model': recovery.BASE_MODEL, 'base_revision': recovery.REVISION, 'examples': 633}
    for key, value in expected.items():
        if metadata.get(key) != value:
            raise ValueError(f'API provenance mismatch: {key}')
    return metadata
read_provenance('http://127.0.0.1:8000/v1')
for attempt in range(6):
    try:
        metadata = read_provenance(BASE_URL)
        break
    except ValueError:
        raise
    except Exception:
        if attempt == 5:
            raise
        time.sleep(5)
(WORK / 'served-model-provenance.json').write_text(json.dumps(metadata, indent=2))
print('LOCAL + PUBLIC API OK. Gửi agent:', BASE_URL)
print('Giữ runtime hoạt động trong lượt đánh giá. Không cần gọi thử model nhiều lần.')

## Cell 7 — Log/bàn giao
Khi agent xác nhận đo xong mới dừng phiên. Mất runtime không mất adapter đã backup local;
merged model và thư viện trong `/content` cần dựng lại ở runtime mới.
Chưa có bằng chứng cần training thêm. Nếu cần nâng cấp, agent sẽ báo lý do và kế hoạch riêng.

In [ ]:
#@title 7. Xem trạng thái và phần cuối log
print('Process:', recovery.managed_process(WORK))
print('Model:', MERGED)
print('Base URL:', globals().get('BASE_URL', 'Chưa chạy cell 5'))
print('\n'.join((WORK / 'api.log').read_text(errors='replace').splitlines()[-20:]) if (WORK / 'api.log').exists() else 'Chưa có api.log')

## Cell 8 — Chỉ dừng sau khi đã đo xong
**Mặc định tắt**, nên Run all không tự dừng API vừa mở. Khi agent xác nhận xong, bật `DO_STOP` và chạy cell.
Cell dừng process model/tunnel của notebook này, giữ các file. Muốn ngừng tính units, sau đó dùng menu Colab
**Runtime → Disconnect and delete runtime** khi đã chắc backup ZIP nằm trên máy/Drive; thao tác đó xóa dữ liệu runtime.

In [ ]:
#@title 8. Dừng API (mặc định bỏ qua)
DO_STOP = False #@param {type:"boolean"}
if DO_STOP:
    recovery.stop_api(WORK)
    print('Đã dừng process API/tunnel của notebook. File model vẫn còn cho tới khi runtime bị xóa.')
else:
    print('Bỏ qua; API tiếp tục phục vụ agent.')